In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data.csv.zip to data.csv (1).zip


In [ ]:
# 1. DataFrame aur Series basics
data = {
    "name": ["aman", "raj", "sahil"],
    "age": [21, 22, 23],
    "course": ["DA", "DS", "AI"],
}
df1 = pd.DataFrame(data)
print(df1)

s = pd.Series([10, 20, 30], name="scores")
print(s)

print(type(df1))
print(type(s))

# 2. Zip CSV file read karna
df = pd.read_csv("data.csv.zip", compression="zip", encoding="latin1")

# 3. Basic inspection functions
print("\n--- First 3 rows ---")
print(df.head(3))

print("\n--- Last 5 rows ---")
print(df.tail())

print("\n--- Random 5 rows ---")
print(df.sample(5, random_state=42))

print("\n--- Shape, Columns, Dtypes ---")
print(df.shape)
print(df.columns)
print(df.dtypes)

print("\n--- Info ---")
print(df.info())

print("\n--- Describe ---")
print(df.describe())
print(df.describe(include="all"))

    name  age course
0   aman   21     DA
1    raj   22     DS
2  sahil   23     AI
0    10
1    20
2    30
Name: scores, dtype: int64
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>

--- First 3 rows ---
  InvoiceNo StockCode                         Description  Quantity  \
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                 WHITE METAL LANTERN         6   
2    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  

--- Last 5 rows ---
       InvoiceNo StockCode                      Description  Quantity  \
541904    581587     22613      PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899     CHILDREN'S APRON DOLLY GIRL          6   
541906  

In [ ]:
# 1. Missing values aur duplicates check karna
print("--- Missing values per column ---")
print(df.isna().sum())

print("\n--- Duplicate rows count ---")
print(df.duplicated().sum())

# 2. Unique values aur value_counts
print("\n--- Unique counts ---")
print("Unique Invoices:", df["InvoiceNo"].nunique())
print("Unique Countries:", df["Country"].nunique())
print("Unique Countries list:", df["Country"].unique())

print("\n--- Top 10 Countries by rows ---")
print(df["Country"].value_counts().head(10))

print("\n--- Top 10 Products by frequency ---")
print(df["Description"].value_counts().head(10))

# 3. Data Cleaning (ek clean copy 'df_clean' mein)
df_clean = df.copy()

# Exact duplicate rows hatana
df_clean = df_clean.drop_duplicates()

# Missing CustomerID aur Description hatana
df_clean = df_clean.dropna(subset=["CustomerID", "Description"])

# InvoiceDate ko datetime mein convert karna
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

# Cancelled invoices ('C' se shuru hone wale InvoiceNo) aur negative/zero Quantity & UnitPrice hatana
df_clean = df_clean[
    (~df_clean["InvoiceNo"].astype(str).str.startswith("C"))
    & (df_clean["Quantity"] > 0)
    & (df_clean["UnitPrice"] > 0)
]

# TotalAmount column banana
df_clean["TotalAmount"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print("\n--- After Cleaning ---")
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Remaining missing values:\n", df_clean.isna().sum())
print("Remaining duplicates:", df_clean.duplicated().sum())

--- Missing values per column ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

--- Duplicate rows count ---
5268

--- Unique counts ---
Unique Invoices: 25900
Unique Countries: 38
Unique Countries list: ['United Kingdom' 'France' 'Australia' 'Netherlands' 'Germany' 'Norway'
 'EIRE' 'Switzerland' 'Spain' 'Poland' 'Portugal' 'Italy' 'Belgium'
 'Lithuania' 'Japan' 'Iceland' 'Channel Islands' 'Denmark' 'Cyprus'
 'Sweden' 'Austria' 'Israel' 'Finland' 'Bahrain' 'Greece' 'Hong Kong'
 'Singapore' 'Lebanon' 'United Arab Emirates' 'Saudi Arabia'
 'Czech Republic' 'Canada' 'Unspecified' 'Brazil' 'USA'
 'European Community' 'Malta' 'RSA']

--- Top 10 Countries by rows ---
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Swi

In [ ]:
# 1. Month aur Year columns nikalna
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.to_period("M")

# 2. Country-wise revenue aur order summary (groupby + agg)
country_summary = (
    df_clean.groupby("Country")
    .agg(
        total_revenue=("TotalAmount", "sum"),
        total_quantity=("Quantity", "sum"),
        unique_orders=("InvoiceNo", "nunique"),
        unique_customers=("CustomerID", "nunique"),
    )
    .sort_values("total_revenue", ascending=False)
)

print("--- Top 10 Countries by Revenue ---")
print(country_summary.head(10))

# 3. Top 10 Products by Total Revenue
top_products_rev = (
    df_clean.groupby("Description")
    .agg(
        total_revenue=("TotalAmount", "sum"),
        total_quantity=("Quantity", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

print("\n--- Top 10 Products by Revenue ---")
print(top_products_rev)

# 4. Month-wise Revenue Trend
monthly_sales = (
    df_clean.groupby("Month")
    .agg(
        monthly_revenue=("TotalAmount", "sum"),
        orders=("InvoiceNo", "nunique"),
    )
)

print("\n--- Monthly Revenue Trend ---")
print(monthly_sales)

# 5. Clean dataset CSV save karna (optional)
df_clean.to_csv("cleaned_ecommerce_data.csv", index=False)
print("\nSaved cleaned_ecommerce_data.csv successfully!")

--- Top 10 Countries by Revenue ---
                total_revenue  total_quantity  unique_orders  unique_customers
Country                                                                       
United Kingdom    7285024.644         4241305          16646              3920
Netherlands        285446.340          200361             94                 9
EIRE               265262.460          140133            260                 3
Germany            228678.400          119154            457                94
France             208934.310          111428            389                87
Australia          138453.810           83891             57                 9
Spain               61558.560           27933             90                30
Switzerland         56443.950           30082             51                21
Belgium             41196.340           23237             98                25
Sweden              38367.830           36078             36                 8

--- Top 10 Prod